In [4]:
from dataclasses import dataclass, field
from pathlib import Path
from typing import List


In [5]:
import os
os.getcwd()


'/home/suyash/codes/PINN'

In [2]:
os.chdir("../")

In [6]:
@dataclass
class PINNArchConfig:
    n_hidden_layers: int
    neurons_per_layer: int
    activation: str 
    n_collocation_points: int 
    n_anchor_points: int 
    n_initial_condition_points: int 
    n_boundary_points: int 
    training_window_days: int 

In [7]:
import yaml
def _load_yaml(path: Path) -> dict:
    with open(path, "r") as f:
        return yaml.safe_load(f)
params_path = Path("params.yaml")
print(f"Loading parameters from {params_path}")
params = _load_yaml(params_path)
print(params)

Loading parameters from params.yaml
{'architecture': {'n_hidden_layers': 7, 'neurons_per_layer': 64, 'activation': 'tanh'}, 'domain': {'depth_m': 30.0, 'training_window_days': 15, 'node_spacing_m': 0.05}, 'sampling': {'n_collocation': 12000, 'n_anchor': 750, 'n_initial_condition': 150, 'n_boundary': 200, 'n_failure_plane_pts': 10}, 'geotechnical': {'Ks': 1.5e-05, 'theta_s': 0.45, 'theta_r': 0.05, 'alpha': 0.8, 'n': 1.4, 'l': 0.5, 'c_prime': 5.0, 'phi_prime': 28.0, 'gamma': 20.0, 'beta': 42.0, 'depth': 25.0}, 'loss_weights': {'lambda_physics': 1.0, 'lambda_anchor': 10.0, 'lambda_initial': 5.0, 'lambda_boundary': 5.0, 'lambda_failure': 20.0}}


In [8]:
def get_pinn_arch_config() -> PINNArchConfig:
    params_path = Path("params.yaml")
    print(f"Loading parameters from {params_path}")
    params = _load_yaml(params_path)
    a = params['architecture']
    d = params['domain']
    s = params['sampling']
    return PINNArchConfig(
            n_hidden_layers=a["n_hidden_layers"],
            neurons_per_layer=a["neurons_per_layer"],
            activation=a["activation"],
            n_collocation_points=s["n_collocation"],
            n_anchor_points=s["n_anchor"],
            n_initial_condition_points=s["n_initial_condition"],
            n_boundary_points=s["n_boundary"],
            training_window_days=d["training_window_days"],
        )
print(get_pinn_arch_config())

Loading parameters from params.yaml
PINNArchConfig(n_hidden_layers=7, neurons_per_layer=64, activation='tanh', n_collocation_points=12000, n_anchor_points=750, n_initial_condition_points=150, n_boundary_points=200, training_window_days=15)


In [7]:
import torch 
import torch.nn as nn
from  typing import Tuple


In [ ]:
import torch
import torch.nn as nn
from typing import Tuple
from dataclasses import dataclass



class PINN(nn.Module):
    def __init__(self, config: PINNArchConfig, input_dim: int = 2, output_dim: int = 1):
        super().__init__()
        self.config = config
        
        # 1. FIX: Define dimensions before building the network
        self.input_dim = input_dim
        self.output_dim = output_dim
        
        self.network = self.build_network()
        
        # 2. FIX: Correct method name reference
        self._init_weights()

    def build_network(self) -> nn.Sequential:  
        act_fn = self._get_activation()
        layers = []
        layers.append(nn.Linear(self.input_dim, self.config.neurons_per_layer))
        layers.append(act_fn())

        for _ in range(self.config.n_hidden_layers):
            layers.append(nn.Linear(self.config.neurons_per_layer, self.config.neurons_per_layer))
            layers.append(act_fn())

        layers.append(nn.Linear(self.config.neurons_per_layer, self.output_dim))
        return nn.Sequential(*layers)
    
    def _get_activation(self):
        mapping = {
            "tanh": nn.Tanh,
            "relu": nn.ReLU,
            "silu": nn.SiLU,
            "gelu": nn.GELU,
        }
        # 3. FIX: Changed self.cfg to self.config
        act = self.config.activation.lower()
        if act not in mapping:
            raise ValueError(f"Unsupported activation: {act}. Choose from {list(mapping)}")
        return mapping[act]

    def _init_weights(self) -> None:
        """Xavier (Glorot) initialisation — standard for PINNs with tanh."""
        for layer in self.network:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_normal_(layer.weight)
                nn.init.zeros_(layer.bias)
    
    # ── Forward pass ──────────────────────────────────────────────────────────
    def forward(self, z_norm: torch.Tensor, t_norm: torch.Tensor) -> torch.Tensor:
        x = torch.cat([z_norm, t_norm], dim=1)
        return self.network(x)

    # ── Convenience: predict with gradients for PDE residual ─────────────────
    def predict_with_gradients(
        self, z_norm: torch.Tensor, t_norm: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        
        z_norm = z_norm.requires_grad_(True)
        t_norm = t_norm.requires_grad_(True)

        psi = self.forward(z_norm, t_norm)

        dpsi_dz = torch.autograd.grad(
            psi, z_norm,
            grad_outputs=torch.ones_like(psi),
            create_graph=True, retain_graph=True,
        )[0]

        dpsi_dt = torch.autograd.grad(
            psi, t_norm,
            grad_outputs=torch.ones_like(psi),
            create_graph=True, retain_graph=True,
        )[0]

        d2psi_dz2 = torch.autograd.grad(
            dpsi_dz, z_norm,
            grad_outputs=torch.ones_like(dpsi_dz),
            create_graph=True, retain_graph=True,
        )[0]

        return psi, dpsi_dz, dpsi_dt, d2psi_dz2




In [10]:
if __name__ == "__main__":
    print("--- Testing PINN Architecture ---")
    
    # 1. Instantiate the Mock Config and Model
    mock_config = PINNArchConfig(neurons_per_layer=64, n_hidden_layers=6, activation="tanh")
    model = PINN(config=mock_config)
    
    # 2. Print the Model Architecture
    print("\n[1] Model Structure:")
    print(model)
    
    # 3. Check Total Trainable Parameters
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n[2] Total Trainable Parameters: {total_params}")
    
    # 4. Generate Dummy Data (e.g., a batch of 5 collocation points)
    # Shapes must be (N, 1) to match your forward pass concatenation logic
    batch_size = 5
    dummy_z = torch.rand((batch_size, 1))
    dummy_t = torch.rand((batch_size, 1))
    
    print("\n[3] Testing Forward Pass...")
    psi_pred = model(dummy_z, dummy_t)
    print(f"Input z shape: {dummy_z.shape}, Input t shape: {dummy_t.shape}")
    print(f"Output psi shape: {psi_pred.shape}")
    assert psi_pred.shape == (batch_size, 1), "Forward pass output shape mismatch!"
    
    print("\n[4] Testing Gradient Computation...")
    psi, dpsi_dz, dpsi_dt, d2psi_dz2 = model.predict_with_gradients(dummy_z, dummy_t)
    print(f"dpsi_dz shape: {dpsi_dz.shape}")
    print(f"dpsi_dt shape: {dpsi_dt.shape}")
    print(f"d2psi_dz2 shape: {d2psi_dz2.shape}")
    print("\n✅ All tests passed successfully!")

--- Testing PINN Architecture ---

[1] Model Structure:
PINN(
  (network): Sequential(
    (0): Linear(in_features=2, out_features=64, bias=True)
    (1): Tanh()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): Tanh()
    (4): Linear(in_features=64, out_features=64, bias=True)
    (5): Tanh()
    (6): Linear(in_features=64, out_features=64, bias=True)
    (7): Tanh()
    (8): Linear(in_features=64, out_features=64, bias=True)
    (9): Tanh()
    (10): Linear(in_features=64, out_features=64, bias=True)
    (11): Tanh()
    (12): Linear(in_features=64, out_features=64, bias=True)
    (13): Tanh()
    (14): Linear(in_features=64, out_features=1, bias=True)
  )
)

[2] Total Trainable Parameters: 25217

[3] Testing Forward Pass...
Input z shape: torch.Size([5, 1]), Input t shape: torch.Size([5, 1])
Output psi shape: torch.Size([5, 1])

[4] Testing Gradient Computation...
dpsi_dz shape: torch.Size([5, 1])
dpsi_dt shape: torch.Size([5, 1])
d2psi_dz2 shape: torch.Size([5, 1